# 09 Validation Strategy Comparison

This notebook compares two internal validation strategies for the selected XGBoost model.

The stratified cross-validation result is loaded from the model comparison notebook. A second XGBoost model is then tuned with expanding-window time-aware validation using the same full feature set and evaluated on the same fixed temporal holdout test set. The notebook produces the validation-strategy comparison table and the selected hyperparameter summary used in the thesis.


## 1. Experiment overview

This section summarizes the validation-strategy comparison setup.

- **Input data:** temporal training and holdout files generated by the split notebook
- **Reference result:** XGBoost result saved by the model comparison notebook
- **Model:** XGBoost
- **Feature set:** same full feature set used in the model comparison
- **Baseline validation strategy:** stratified cross-validation result loaded from the model comparison output
- **Alternative validation strategy:** expanding-window time-aware validation within the training set
- **Final evaluation:** fixed temporal holdout test set
- **Selection metric:** F1-score
- **Threshold:** 0.5 for threshold-dependent metrics


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import joblib
import platform
import warnings

import numpy as np
import pandas as pd
import sklearn
import xgboost

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)


## 2. Project paths and settings

This section defines repository-style input and output paths.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
LOGS_DIR = OUTPUTS_DIR / "logs"
MODELS_DIR = OUTPUTS_DIR / "models"

MODEL_COMPARISON_TABLES_DIR = TABLES_DIR / "model_comparison"
VALIDATION_TABLES_DIR = TABLES_DIR / "validation_strategy_comparison"
VALIDATION_LOG_DIR = LOGS_DIR / "validation_strategy_comparison"
VALIDATION_MODEL_DIR = MODELS_DIR / "validation_strategy_comparison"

X_TRAIN_PATH = INTERIM_DIR / "X_train_temporal.csv"
X_TEST_PATH = INTERIM_DIR / "X_test_temporal.csv"
Y_TRAIN_PATH = INTERIM_DIR / "y_train_temporal.csv"
Y_TEST_PATH = INTERIM_DIR / "y_test_temporal.csv"

MODEL_COMPARISON_RESULTS_PATH = MODEL_COMPARISON_TABLES_DIR / "model_comparison_results.csv"
MODEL_COMPARISON_BEST_PARAMS_PATH = MODEL_COMPARISON_TABLES_DIR / "model_comparison_best_params.csv"
MODEL_COMPARISON_SELECTED_FEATURES_PATH = MODEL_COMPARISON_TABLES_DIR / "model_comparison_selected_features.csv"

TARGET_COL = "target"
TEMPORAL_SPLIT_COL = "last_funding_at"
MODEL_NAME = "XGBoost"

RANDOM_SEED = 2026
TIME_AWARE_CV_FOLDS = 5
THRESHOLD = 0.5
SCORING = "f1"

CATEGORICAL_COLUMNS = ["country_code", "market"]

GRIDSEARCH_N_JOBS = -1
GRIDSEARCH_VERBOSE = 1

OUTPUT_PATHS = {
    "comparison": VALIDATION_TABLES_DIR / "validation_strategy_comparison_results.csv",
    "best_params": VALIDATION_TABLES_DIR / "validation_strategy_comparison_best_params.csv",
    "time_aware_predictions": VALIDATION_TABLES_DIR / "xgboost_time_aware_holdout_predictions.csv",
    "time_aware_fold_summary": VALIDATION_TABLES_DIR / "time_aware_fold_summary.csv",
    "time_aware_gridsearch_log": VALIDATION_LOG_DIR / "xgboost_time_aware_gridsearch_log.csv",
    "time_aware_model": VALIDATION_MODEL_DIR / "xgboost_time_aware_best_estimator.joblib",
    "run_metadata": VALIDATION_TABLES_DIR / "validation_strategy_comparison_run_metadata.json",
}

for directory in [VALIDATION_TABLES_DIR, VALIDATION_LOG_DIR, VALIDATION_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## 3. Hyperparameter search space

This section defines the XGBoost search space used for the time-aware validation setting.


In [3]:
XGB_PARAMS = {
    "model__n_estimators": [100, 300, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 1.0],
    "model__colsample_bytree": [0.7, 1.0],
}

parameter_summary = pd.DataFrame({
    "model": [MODEL_NAME],
    "n_parameter_settings": [int(np.prod([len(values) for values in XGB_PARAMS.values()]))],
})

display(parameter_summary)


,model,n_parameter_settings
0,XGBoost,324


## 4. Helper functions

This section defines reusable functions for loading data, constructing the pipeline, creating time-aware folds, and evaluating holdout performance.


In [4]:
def display_table(df: pd.DataFrame) -> None:
    display(df.round(2))


def check_required_files(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "The following required input files are missing:\n"
            + "\n".join(missing)
            + "\n\nRun the temporal split and model comparison notebooks first."
        )


def read_target(path: Path, target_col: str = TARGET_COL) -> pd.Series:
    target_df = pd.read_csv(path)
    if target_col not in target_df.columns:
        raise ValueError(f"Target column '{target_col}' was not found in {path}.")
    return target_df[target_col].astype(int)


def load_temporal_split_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    check_required_files([X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH])
    X_train = pd.read_csv(X_TRAIN_PATH)
    X_test = pd.read_csv(X_TEST_PATH)
    y_train = read_target(Y_TRAIN_PATH)
    y_test = read_target(Y_TEST_PATH)
    return X_train, X_test, y_train, y_test


def load_selected_features(path: Path) -> list[str]:
    selected_features_df = pd.read_csv(path)

    if "feature" in selected_features_df.columns:
        feature_column = "feature"
    elif "selected_feature" in selected_features_df.columns:
        feature_column = "selected_feature"
    else:
        raise ValueError(
            "Expected a column named 'feature' or 'selected_feature' in the selected-feature table. "
            f"Available columns: {list(selected_features_df.columns)}"
        )

    return selected_features_df[feature_column].dropna().astype(str).tolist()


def load_xgboost_reference_result() -> pd.DataFrame:
    check_required_files([MODEL_COMPARISON_RESULTS_PATH])
    model_comparison_results = pd.read_csv(MODEL_COMPARISON_RESULTS_PATH)

    if "model" not in model_comparison_results.columns:
        raise ValueError("The model comparison results file must contain a 'model' column.")

    if "f1" not in model_comparison_results.columns:
        raise ValueError("The model comparison results file must contain an 'f1' column.")

    model_mask = (
        model_comparison_results["model"].astype(str).str.strip().str.lower()
        == MODEL_NAME.strip().lower()
    )
    xgb_rows = model_comparison_results.loc[model_mask].copy()

    if xgb_rows.empty:
        available_models = sorted(model_comparison_results["model"].dropna().astype(str).unique())
        raise ValueError(
            f"No row was found for model '{MODEL_NAME}'. "
            f"Available models: {available_models}"
        )

    best_model_name = model_comparison_results.sort_values("f1", ascending=False).iloc[0]["model"]
    if str(best_model_name).strip().lower() != MODEL_NAME.strip().lower():
        raise ValueError(
            f"The highest holdout F1-score is currently from '{best_model_name}', "
            f"not '{MODEL_NAME}'."
        )

    xgb_rows["validation_strategy"] = "Stratified CV"
    return xgb_rows


def load_xgboost_stratified_best_params() -> pd.DataFrame:
    check_required_files([MODEL_COMPARISON_BEST_PARAMS_PATH])
    best_params = pd.read_csv(MODEL_COMPARISON_BEST_PARAMS_PATH)

    if "model" not in best_params.columns:
        raise ValueError("The best-parameter table must contain a 'model' column.")

    model_mask = (
        best_params["model"].astype(str).str.strip().str.lower()
        == MODEL_NAME.strip().lower()
    )

    xgb_best_params = best_params.loc[model_mask].copy()
    if xgb_best_params.empty:
        available_models = sorted(best_params["model"].dropna().astype(str).unique())
        raise ValueError(
            f"No best-parameter rows were found for model '{MODEL_NAME}'. "
            f"Available models: {available_models}"
        )

    xgb_best_params["validation_strategy"] = "Stratified CV"
    return xgb_best_params


def select_columns(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    selected_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    present = [column for column in selected_columns if column in X_train.columns and column in X_test.columns]
    missing = [column for column in selected_columns if column not in X_train.columns or column not in X_test.columns]

    if not present:
        raise ValueError("None of the selected features were found in both X_train and X_test.")

    return X_train[present].copy(), X_test[present].copy(), present, missing


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def make_preprocessor(X_train: pd.DataFrame, categorical_columns: list[str]) -> ColumnTransformer:
    categorical_present = [column for column in categorical_columns if column in X_train.columns]
    numeric_present = [column for column in X_train.columns if column not in categorical_present]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="other")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_present),
            ("categorical", categorical_pipeline, categorical_present),
        ],
        remainder="drop",
    )


def make_xgboost_pipeline(X_train: pd.DataFrame, categorical_columns: list[str]) -> Pipeline:
    return Pipeline([
        ("preprocessor", make_preprocessor(X_train, categorical_columns)),
        ("model", XGBClassifier(
            random_state=RANDOM_SEED,
            eval_metric="logloss",
            n_jobs=1,
            tree_method="hist",
        )),
    ])


def make_time_aware_cv_folds(
    X_train_raw: pd.DataFrame,
    y_train: pd.Series,
    split_col: str = TEMPORAL_SPLIT_COL,
    n_splits: int = TIME_AWARE_CV_FOLDS,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], pd.DataFrame]:
    if split_col not in X_train_raw.columns:
        raise ValueError(f"Temporal split column '{split_col}' was not found in X_train.")

    fold_frame = pd.DataFrame({
        "position": np.arange(len(X_train_raw)),
        "split_date": pd.to_datetime(X_train_raw[split_col], errors="coerce"),
        "target": y_train.to_numpy(),
    })

    if fold_frame["split_date"].isna().any():
        missing_count = int(fold_frame["split_date"].isna().sum())
        raise ValueError(f"{missing_count} rows have missing or invalid {split_col} values.")

    fold_frame = fold_frame.sort_values(["split_date", "position"]).reset_index(drop=True)
    blocks = np.array_split(fold_frame["position"].to_numpy(), n_splits + 1)

    cv_folds = []
    summary_rows = []

    for fold_id in range(1, n_splits + 1):
        train_idx = np.concatenate(blocks[:fold_id])
        valid_idx = blocks[fold_id]

        y_train_fold = y_train.iloc[train_idx]
        y_valid_fold = y_train.iloc[valid_idx]

        if y_train_fold.nunique() < 2:
            raise ValueError(f"Fold {fold_id} training partition contains only one class.")
        if y_valid_fold.nunique() < 2:
            raise ValueError(f"Fold {fold_id} validation partition contains only one class.")

        train_dates = pd.to_datetime(X_train_raw.iloc[train_idx][split_col])
        valid_dates = pd.to_datetime(X_train_raw.iloc[valid_idx][split_col])

        summary_rows.append({
            "fold": fold_id,
            "train_size": len(train_idx),
            "valid_size": len(valid_idx),
            "train_period": f"{train_dates.min().date()}–{train_dates.max().date()}",
            "validation_period": f"{valid_dates.min().date()}–{valid_dates.max().date()}",
        })

        cv_folds.append((train_idx, valid_idx))

    return cv_folds, pd.DataFrame(summary_rows)


def evaluate_binary_classifier(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    threshold: float = THRESHOLD,
) -> tuple[dict, pd.DataFrame]:
    y_prob = fitted_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

    predictions = pd.DataFrame({
        "model": MODEL_NAME,
        "validation_strategy": "Time-aware CV",
        "y_true": y_test.to_numpy(),
        "y_pred": y_pred,
        "y_prob": y_prob,
    })

    return metrics, predictions


def summarize_best_params(model_name: str, validation_strategy: str, search: GridSearchCV) -> pd.DataFrame:
    rows = []
    for parameter, compared_values in search.param_grid.items():
        rows.append({
            "model": model_name,
            "validation_strategy": validation_strategy,
            "parameter": parameter.replace("model__", ""),
            "compared_values": ", ".join(map(str, compared_values)),
            "best_value": search.best_params_.get(parameter),
        })
    return pd.DataFrame(rows)


## 5. Load inputs and reference result

This section loads the temporal split files and the XGBoost result from the model comparison output.


In [5]:
required_paths = [
    X_TRAIN_PATH,
    X_TEST_PATH,
    Y_TRAIN_PATH,
    Y_TEST_PATH,
    MODEL_COMPARISON_RESULTS_PATH,
    MODEL_COMPARISON_BEST_PARAMS_PATH,
    MODEL_COMPARISON_SELECTED_FEATURES_PATH,
]

check_required_files(required_paths)

X_train_raw, X_test_raw, y_train, y_test = load_temporal_split_inputs()
xgboost_reference = load_xgboost_reference_result()

input_summary = pd.DataFrame({
    "item": [
        "training_observations",
        "holdout_observations",
        "training_positive_count",
        "training_negative_count",
        "holdout_positive_count",
        "holdout_negative_count",
    ],
    "value": [
        X_train_raw.shape[0],
        X_test_raw.shape[0],
        int((y_train == 1).sum()),
        int((y_train == 0).sum()),
        int((y_test == 1).sum()),
        int((y_test == 0).sum()),
    ],
})

display_table(input_summary)
display_table(xgboost_reference)


,item,value
0,training_observations,4563
1,holdout_observations,1141
2,training_positive_count,2606
3,training_negative_count,1957
4,holdout_positive_count,781
5,holdout_negative_count,360


,model,f1,precision,recall,accuracy,roc_auc,true_negative,false_positive,false_negative,true_positive,validation_strategy
0,XGBoost,0.81,0.79,0.82,0.73,0.77,193,167,139,642,Stratified CV


## 6. Select the full feature set

This section uses the same selected features saved by the model comparison notebook.


In [6]:
selected_feature_candidates = load_selected_features(MODEL_COMPARISON_SELECTED_FEATURES_PATH)

X_train, X_test, selected_features, missing_features = select_columns(
    X_train_raw,
    X_test_raw,
    selected_feature_candidates,
)

if missing_features:
    raise ValueError(f"Expected features missing from the split files: {missing_features}")

selected_feature_table = pd.DataFrame({"feature": selected_features})

display_table(pd.DataFrame({
    "item": ["selected_features"],
    "value": [len(selected_features)],
}))
display(selected_feature_table)


,item,value
0,selected_features,19


,feature
0,funding_total_usd
1,seed
2,venture
3,angel
4,debt_financing
5,private_equity
6,undisclosed
7,founded_year
8,funding_rounds
9,funding_duration_days


## 7. Create time-aware validation folds

This section creates expanding-window validation folds inside the training set.


In [7]:
time_aware_cv_folds, time_aware_fold_summary = make_time_aware_cv_folds(
    X_train_raw=X_train_raw,
    y_train=y_train,
    split_col=TEMPORAL_SPLIT_COL,
    n_splits=TIME_AWARE_CV_FOLDS,
)

display(time_aware_fold_summary)


,fold,train_size,valid_size,train_period,validation_period
0,1,761,761,1993-08-30–2006-11-01,2006-11-02–2008-02-04
1,2,1522,761,1993-08-30–2008-02-04,2008-02-04–2009-04-30
2,3,2283,760,1993-08-30–2009-04-30,2009-05-01–2010-04-02
3,4,3043,760,1993-08-30–2010-04-02,2010-04-03–2011-02-12
4,5,3803,760,1993-08-30–2011-02-12,2011-02-14–2012-01-13


## 8. Tune XGBoost with time-aware validation

This section performs grid search with the time-aware validation folds.


In [8]:
xgboost_time_aware_pipeline = make_xgboost_pipeline(X_train, CATEGORICAL_COLUMNS)

xgboost_time_aware_search = GridSearchCV(
    estimator=xgboost_time_aware_pipeline,
    param_grid=XGB_PARAMS,
    scoring=SCORING,
    cv=time_aware_cv_folds,
    n_jobs=GRIDSEARCH_N_JOBS,
    verbose=GRIDSEARCH_VERBOSE,
    refit=True,
    return_train_score=True,
)

xgboost_time_aware_search.fit(X_train, y_train)

time_aware_cv_summary = pd.DataFrame({
    "model": [MODEL_NAME],
    "validation_strategy": ["Time-aware CV"],
    "best_cv_f1": [float(xgboost_time_aware_search.best_score_)],
})

display_table(time_aware_cv_summary)


Fitting 5 folds for each of 324 candidates, totalling 1620 fits


,model,validation_strategy,best_cv_f1
0,XGBoost,Time-aware CV,0.72


## 9. Evaluate the time-aware selected model

This section evaluates the time-aware selected XGBoost model on the fixed temporal holdout test set.


In [9]:
time_aware_metrics, time_aware_predictions = evaluate_binary_classifier(
    fitted_model=xgboost_time_aware_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    threshold=THRESHOLD,
)

time_aware_metrics["model"] = MODEL_NAME
time_aware_metrics["validation_strategy"] = "Time-aware CV"

time_aware_result = pd.DataFrame([time_aware_metrics])

display_table(time_aware_result)


,f1,precision,recall,accuracy,roc_auc,true_negative,false_positive,false_negative,true_positive,model,validation_strategy
0,0.80,0.80,0.80,0.73,0.76,207,153,157,624,XGBoost,Time-aware CV


## 10. Compare validation strategies

This section combines the stratified-validation result from the model comparison notebook with the time-aware validation result from this notebook.


In [10]:
comparison_columns = [
    "validation_strategy",
    "f1",
    "precision",
    "recall",
    "accuracy",
    "roc_auc",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
]

stratified_reference = xgboost_reference.copy()

for column in comparison_columns:
    if column not in stratified_reference.columns:
        stratified_reference[column] = np.nan

stratified_reference = stratified_reference.loc[:, comparison_columns]
time_aware_result_for_comparison = time_aware_result.loc[:, comparison_columns]

validation_comparison_df = pd.concat(
    [stratified_reference, time_aware_result_for_comparison],
    ignore_index=True,
)

validation_comparison_table = validation_comparison_df[[
    "validation_strategy",
    "f1",
    "precision",
    "recall",
    "accuracy",
    "roc_auc",
]].copy()

metric_difference = (
    validation_comparison_df
    .set_index("validation_strategy")
    .loc["Time-aware CV", ["f1", "precision", "recall", "accuracy", "roc_auc"]]
    - validation_comparison_df
    .set_index("validation_strategy")
    .loc["Stratified CV", ["f1", "precision", "recall", "accuracy", "roc_auc"]]
)

metric_difference_df = (
    metric_difference
    .rename("time_aware_minus_stratified")
    .reset_index()
    .rename(columns={"index": "metric"})
)

display_table(validation_comparison_table)
display_table(metric_difference_df)


,validation_strategy,f1,precision,recall,accuracy,roc_auc
0,Stratified CV,0.81,0.79,0.82,0.73,0.77
1,Time-aware CV,0.80,0.80,0.80,0.73,0.76


,metric,time_aware_minus_stratified
0,f1,-0.01
1,precision,0.01
2,recall,-0.02
3,accuracy,-0.00
4,roc_auc,-0.00


## 11. Summarize selected hyperparameters

This section combines the selected XGBoost hyperparameters under stratified and time-aware validation.


In [11]:
stratified_best_params_df = load_xgboost_stratified_best_params()

time_aware_best_params_df = summarize_best_params(
    model_name=MODEL_NAME,
    validation_strategy="Time-aware CV",
    search=xgboost_time_aware_search,
)

validation_best_params_df = pd.concat(
    [stratified_best_params_df, time_aware_best_params_df],
    ignore_index=True,
)

display(validation_best_params_df)


,model,parameter,compared_values,best_value,validation_strategy
0,XGBoost,n_estimators,"100, 300, 500",100,Stratified CV
1,XGBoost,learning_rate,"0.01, 0.05, 0.1",0.01,Stratified CV
2,XGBoost,max_depth,"3, 5, 7",7,Stratified CV
3,XGBoost,min_child_weight,"1, 3, 5",1,Stratified CV
4,XGBoost,subsample,"0.7, 1.0",0.7,Stratified CV
5,XGBoost,colsample_bytree,"0.7, 1.0",0.7,Stratified CV
6,XGBoost,n_estimators,"100, 300, 500",100.00,Time-aware CV
7,XGBoost,learning_rate,"0.01, 0.05, 0.1",0.01,Time-aware CV
8,XGBoost,max_depth,"3, 5, 7",5.00,Time-aware CV
9,XGBoost,min_child_weight,"1, 3, 5",1.00,Time-aware CV


## 12. Save outputs

This section saves the validation-strategy comparison results and related reproducibility outputs.


In [12]:
gridsearch_log_df = pd.DataFrame(xgboost_time_aware_search.cv_results_)

OUTPUT_PATHS["comparison"].parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS["time_aware_gridsearch_log"].parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS["time_aware_model"].parent.mkdir(parents=True, exist_ok=True)

validation_comparison_df.to_csv(OUTPUT_PATHS["comparison"], index=False)
validation_best_params_df.to_csv(OUTPUT_PATHS["best_params"], index=False)
time_aware_predictions.to_csv(OUTPUT_PATHS["time_aware_predictions"], index=False)
time_aware_fold_summary.to_csv(OUTPUT_PATHS["time_aware_fold_summary"], index=False)
gridsearch_log_df.to_csv(OUTPUT_PATHS["time_aware_gridsearch_log"], index=False)
joblib.dump(xgboost_time_aware_search.best_estimator_, OUTPUT_PATHS["time_aware_model"])

run_metadata = {
    "model": MODEL_NAME,
    "target_col": TARGET_COL,
    "temporal_split_col": TEMPORAL_SPLIT_COL,
    "random_seed": RANDOM_SEED,
    "time_aware_cv_folds": TIME_AWARE_CV_FOLDS,
    "scoring": SCORING,
    "threshold": THRESHOLD,
    "selected_features": selected_features,
    "missing_features": missing_features,
    "input_paths": {
        "X_train": str(X_TRAIN_PATH),
        "X_test": str(X_TEST_PATH),
        "y_train": str(Y_TRAIN_PATH),
        "y_test": str(Y_TEST_PATH),
        "model_comparison_results": str(MODEL_COMPARISON_RESULTS_PATH),
        "model_comparison_best_params": str(MODEL_COMPARISON_BEST_PARAMS_PATH),
        "model_comparison_selected_features": str(MODEL_COMPARISON_SELECTED_FEATURES_PATH),
    },
    "output_paths": {key: str(value) for key, value in OUTPUT_PATHS.items()},
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    },
}

with open(OUTPUT_PATHS["run_metadata"], "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
